<a href="https://colab.research.google.com/github/Siraj-AI-Lab/Pakistan-Flood-Prediction-ML/blob/main/Research_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import pandas as pd

# 1. Unzip archive (2).zip if the CSVs are still inside it
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]
for z in zip_files:
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall('.')
    print(f"Extracted {z} successfully!")

# 2. List all available CSV files in Colab working directory
csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
print("\n=== Detected CSV Files ===")
for f in csv_files:
    print(f)

# 3. Automatically load the national dataset
national_file = [f for f in csv_files if 'national' in f.lower()]

if national_file:
    df_national = pd.read_csv(national_file[0])
    print(f"\nSuccessfully loaded: {national_file[0]}")
    print("\n=== National Dataset Columns ===")
    print(df_national.columns.tolist())
    print("\n=== First 5 Rows ===")
    print(df_national.head())
else:
    print("\nNo file matching 'national' found. Check the printed list above!")


=== Detected CSV Files ===

No file matching 'national' found. Check the printed list above!


In [ ]:
import os
import glob
import pandas as pd

# 1. Search for all CSV and Excel files across all directories and subfolders
all_files = []
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith(('.csv', '.xls', '.xlsx')) and not file.startswith('.'):
            all_files.append(os.path.join(root, file))

print("=== All Discovered Data Files ===")
for f in all_files:
    print(f)

# 2. Find and load the national dataset
national_files = [f for f in all_files if 'national' in f.lower()]

if national_files:
    target_file = national_files[0]
    print(f"\nLoading file: {target_file}")

    if target_file.endswith('.csv'):
        df_national = pd.read_csv(target_file)
    else:
        df_national = pd.read_excel(target_file)

    print("\n=== National Dataset Successfully Loaded! ===")
    print("\nColumns:", df_national.columns.tolist())
    print("\nFirst 3 rows:")
    print(df_national.head(3))
else:
    print("\nNo file matching 'national' found. Check the printed list above!")

=== All Discovered Data Files ===
./sample_data/california_housing_test.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_train.csv
./sample_data/mnist_test.csv

No file matching 'national' found. Check the printed list above!


No .zip file detected in current directory.

=== Detected Dataset Files ===

File 'pakistan_flood_damage_national_2010_2025' not found.
If missing, re-upload the file or archive using the left sidebar Upload button (📁 -> ⬆️).


In [ ]:
from google.colab import files
import pandas as pd
import io

# 1. This will open a "Choose Files" button in your notebook output
print("Click the button below to upload your dataset files (CSV or Excel):")
uploaded = files.upload()

# 2. Check if the national dataset was uploaded and load it
target_filename = 'pakistan_flood_damage_national_2010_2025'

# Find the exact uploaded filename that matches our target
actual_file = None
for filename in uploaded.keys():
    if target_filename in filename:
        actual_file = filename
        break

if actual_file:
    print(f"\nSuccessfully found uploaded file: {actual_file}")

    # Load dataset dynamically based on extension
    if actual_file.endswith('.csv'):
        df_national = pd.read_csv(io.BytesIO(uploaded[actual_file]))
    else:
        df_national = pd.read_excel(io.BytesIO(uploaded[actual_file]))

    print("\n=== National Dataset Columns ===")
    print(df_national.columns.tolist())
    print("\n=== First 3 Rows ===")
    print(df_national.head(3))
else:
    print(f"\nError: Could not find '{target_filename}' in the uploaded files.")
    print("Files you uploaded:", list(uploaded.keys()))

Click the button below to upload your dataset files (CSV or Excel):


Saving pakistan_flood_damage_district_level.csv to pakistan_flood_damage_district_level (1).csv
Saving pakistan_flood_damage_national_2010_2025.csv to pakistan_flood_damage_national_2010_2025 (1).csv
Saving pakistan_flood_damage_province_level.csv to pakistan_flood_damage_province_level (1).csv
Saving pakistan_flood_events_timeline.csv to pakistan_flood_events_timeline (1).csv

Successfully found uploaded file: pakistan_flood_damage_national_2010_2025 (1).csv

=== National Dataset Columns ===
['year', 'deaths', 'injuries', 'people_affected_millions', 'displaced_millions', 'houses_destroyed', 'houses_damaged', 'livestock_lost', 'roads_damaged_km', 'bridges_damaged', 'economic_loss_usd_B', 'flooded_area_km2', 'districts_calamity_hit', 'rainfall_anomaly_pct', 'flood_severity', 'primary_cause', 'primary_provinces_hit', 'children_deaths', 'women_deaths', 'schools_damaged', 'health_facilities_damaged', 'cropland_affected_acres', 'data_source', 'data_quality', 'deaths_per_million_affected', '

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt
import shap

# Clean column headers
df_national.columns = df_national.columns.str.strip()

# Select numeric features
numeric_cols = df_national.select_dtypes(include=[np.number]).columns.tolist()

# Identify target variable (Economic Loss / Damage)
target_matches = [col for col in numeric_cols if any(k in col.lower() for k in ['loss', 'economic', 'damage', 'usd'])]
target = target_matches[0] if target_matches else numeric_cols[-1]

print(f"Target Variable Identified: '{target}'")

# Separate features (X) and target (y)
X = df_national[numeric_cols].drop(columns=[target], errors='ignore')
y = df_national[target]

# Handle missing values
X = X.fillna(X.median())
y = y.fillna(y.median())

# 1. Cross-Validation Setup
tscv = TimeSeriesSplit(n_splits=3)
models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost Regressor": xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
}

results = []

for name, model in models.items():
    rmses, maes, r2s = [], [], []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        rmses.append(np.sqrt(mean_squared_error(y_test, preds)))
        maes.append(mean_absolute_error(y_test, preds))
        r2s.append(r2_score(y_test, preds))

    results.append({
        'Model': name,
        'RMSE': np.mean(rmses),
        'MAE': np.mean(maes),
        'R2 Score': np.mean(r2s)
    })

results_df = pd.DataFrame(results)
print("\n=== Model Benchmarking Results ===")
print(results_df.to_string(index=False))

# 2. Train Best Model (XGBoost) & Extract SHAP Plot
best_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
best_model.fit(X, y)

explainer = shap.Explainer(best_model, X)
shap_values = explainer(X)

# Save SHAP figure for paper
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X, show=False)
plt.savefig('shap_summary.png', bbox_inches='tight', dpi=300)
plt.close()

print("\nSHAP feature importance graph saved successfully as 'shap_summary.png'!")

Target Variable Identified: 'houses_damaged'

=== Model Benchmarking Results ===
            Model          RMSE           MAE  R2 Score
 Ridge Regression    588.890064    501.525530  0.999439
    Random Forest 200235.874224 137651.416667 -1.937081
XGBoost Regressor 241256.835651 150507.545573 -0.561542


NotImplementedError: Categorical split is not yet supported. You can still use TreeExplainer with `feature_perturbation=tree_path_dependent`.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt
import shap

# 1. Locate and Load National File Automatically
national_files = glob.glob('*national*.csv') + glob.glob('*national*.xls*')

if not national_files:
    raise FileNotFoundError("National dataset file not found in directory. Please upload 'pakistan_flood_damage_national_2010_2025' file.")

file_path = national_files[0]
print(f"Loading dataset: {file_path}")

df_national = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
df_national.columns = df_national.columns.str.strip()

# 2. Select Numerical Features & Target
numeric_cols = df_national.select_dtypes(include=[np.number]).columns.tolist()

# Exclude year column from feature matrix if present
if 'Year' in numeric_cols:
    numeric_cols.remove('Year')

# Target identification (Economic Loss / Damage)
target_matches = [col for col in numeric_cols if any(k in col.lower() for k in ['loss', 'economic', 'damage', 'usd'])]
target = target_matches[0] if target_matches else numeric_cols[-1]

print(f"Target Variable: '{target}'")

X = df_national[numeric_cols].drop(columns=[target], errors='ignore')
y = df_national[target]

# Handle Missing Values
X = X.fillna(X.median())
y = y.fillna(y.median())

# 3. Model Benchmarking via Time-Series Cross Validation
tscv = TimeSeriesSplit(n_splits=min(3, len(df_national) - 1))
models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost Regressor": xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
}

results = []

for name, model in models.items():
    rmses, maes, r2s = [], [], []
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        rmses.append(np.sqrt(mean_squared_error(y_test, preds)))
        maes.append(mean_absolute_error(y_test, preds))
        r2s.append(r2_score(y_test, preds))

    results.append({
        'Model': name,
        'RMSE': np.mean(rmses),
        'MAE': np.mean(maes),
        'R2 Score': np.mean(r2s)
    })

results_df = pd.DataFrame(results)
print("\n=== Model Benchmarking Results ===")
print(results_df.to_string(index=False))

# 4. Train XGBoost Model & Save SHAP Interpretability Plot
best_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
best_model.fit(X, y)

explainer = shap.Explainer(best_model, X)
shap_values = explainer(X)

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X, show=False)
plt.savefig('shap_summary.png', bbox_inches='tight', dpi=300)
plt.close()

print("\nSHAP feature importance graph saved successfully as 'shap_summary.png'!")

Loading dataset: pakistan_flood_damage_national_2010_2025.csv
Target Variable: 'houses_damaged'

=== Model Benchmarking Results ===
            Model          RMSE           MAE  R2 Score
 Ridge Regression    588.890064    501.525530  0.999439
    Random Forest 200235.874224 137651.416667 -1.937081
XGBoost Regressor 241256.835651 150507.545573 -0.561542


NotImplementedError: Categorical split is not yet supported. You can still use TreeExplainer with `feature_perturbation=tree_path_dependent`.

In [ ]:
# 4. Train XGBoost Model & Save SHAP Interpretability Plot (Fixed Explainer)
best_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
best_model.fit(X, y)

# Direct TreeExplainer call fixes the Python 3.13 compatibility issue
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X, show=False)
plt.savefig('shap_summary.png', bbox_inches='tight', dpi=300)
plt.close()

print("\nSHAP feature importance graph saved successfully as 'shap_summary.png'!")


SHAP feature importance graph saved successfully as 'shap_summary.png'!
